In [1]:
import requests
import json
import pandas as pd
from pathlib import Path
import time

print("Imports successful!")

Imports successful!


In [2]:
drug_name = "warfarin"

url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"

response = requests.get(url, timeout=30)

print("Status code:", response.status_code)
print(response.json())

Status code: 200
{'idGroup': {'rxnormId': ['11289']}}


In [3]:
CURRENT_DIR = Path.cwd()

print("Current working directory:")
print(CURRENT_DIR)

# Notebook is expected to be inside:
# Prescription_safety_graph/notebooks/

PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"

TDC_DIR = RAW_DIR / "tdc"
TWOSIDES_DIR = RAW_DIR / "twosides"
RXNAV_DIR = RAW_DIR / "rxnav"
NORMALIZED_DIR = INTERIM_DIR / "normalized"

print("\nProject root:")
print(PROJECT_ROOT)

print("\nImportant directories:")
for name, path in {
    "TDC": TDC_DIR,
    "TWOSIDES": TWOSIDES_DIR,
    "RXNAV": RXNAV_DIR,
    "NORMALIZED": NORMALIZED_DIR
}.items():
    print(f"{name}: {path} | Exists: {path.exists()}")

Current working directory:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/notebooks

Project root:
/Users/apple/Documents/SIH2026/Prescription_safety_graph

Important directories:
TDC: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/tdc | Exists: True
TWOSIDES: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/twosides | Exists: True
RXNAV: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/rxnav | Exists: True
NORMALIZED: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/interim/normalized | Exists: True


In [4]:
RXNAV_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

print("Required directories are ready:")
print("RXNAV:", RXNAV_DIR.resolve())
print("NORMALIZED:", NORMALIZED_DIR.resolve())

Required directories are ready:
RXNAV: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/rxnav
NORMALIZED: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/interim/normalized


In [5]:
def list_directory(path, title):
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")

    if not path.exists():
        print("Directory does not exist.")
        return

    files = sorted(path.rglob("*"))

    if not files:
        print("Directory is empty.")
        return

    for file in files:
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"{file.relative_to(PROJECT_ROOT)} | {size_mb:.2f} MB")


list_directory(TDC_DIR, "TDC RAW DIRECTORY")
list_directory(TWOSIDES_DIR, "TWOSIDES RAW DIRECTORY")


TDC RAW DIRECTORY
data/raw/tdc/drugbank.tab | 42.33 MB

TWOSIDES RAW DIRECTORY
data/raw/twosides/twosides.csv | 645.90 MB


In [6]:
def find_csv_files(path):
    if not path.exists():
        return []

    return sorted(
        [
            file
            for file in path.rglob("*")
            if file.is_file() and file.suffix.lower() in [".csv", ".tsv"]
        ]
    )


tdc_files = find_csv_files(TDC_DIR)
twosides_files = find_csv_files(TWOSIDES_DIR)

print("\nTDC candidate files:")
for file in tdc_files:
    print(file)

print("\nTWOSIDES candidate files:")
for file in twosides_files:
    print(file)


TDC candidate files:

TWOSIDES candidate files:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/twosides/twosides.csv


In [7]:
def inspect_file_header(file_path, nrows=5):
    print(f"\n{'='*100}")
    print("FILE:", file_path.name)
    print("PATH:", file_path)
    print(f"{'='*100}")

    try:
        if file_path.suffix.lower() == ".tsv":
            sample = pd.read_csv(file_path, sep="\t", nrows=nrows)
        else:
            sample = pd.read_csv(file_path, nrows=nrows)

        print("Columns:")
        print(sample.columns.tolist())

        print("\nFirst rows:")
        print(sample)

    except Exception as e:
        print("ERROR:", e)


for file in tdc_files:
    inspect_file_header(file)

for file in twosides_files:
    inspect_file_header(file)


FILE: twosides.csv
PATH: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/twosides/twosides.csv
Columns:
['ID1', 'ID2', 'Y', 'Side Effect Name', 'X1', 'X2']

First rows:
            ID1           ID2     Y            Side Effect Name  \
0  CID000002173  CID000003345  1024             hypermagnesemia   
1  CID000002173  CID000003345   767  retinopathy of prematurity   
2  CID000002173  CID000003345    79                 atelectasis   
3  CID000002173  CID000003345    25                   alkalosis   
4  CID000002173  CID000003345    85                   Back Ache   

                                                  X1  \
0  CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...   
1  CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...   
2  CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...   
3  CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...   
4  CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...   

                                             X2  
0  CCC(=O)N(C1CCN(CC1)

In [11]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# LOCATE DRUGBANK FILE
# --------------------------------------------------

drugbank_candidates = (
    list(TDC_DIR.glob("*.tab")) +
    list(TDC_DIR.glob("*.csv")) +
    list(TDC_DIR.glob("*.tsv"))
)

print("DrugBank candidate files:")
for file in drugbank_candidates:
    print(file)

if not drugbank_candidates:
    raise FileNotFoundError(
        f"No DrugBank dataset file found inside: {TDC_DIR}"
    )

drugbank_file = drugbank_candidates[0]

print("\nDetected DrugBank file:")
print(drugbank_file)

# --------------------------------------------------
# LOAD BASED ON FILE TYPE
# --------------------------------------------------

if drugbank_file.suffix == ".tab":
    drugbank_df = pd.read_csv(
        drugbank_file,
        sep="\t"
    )

elif drugbank_file.suffix == ".tsv":
    drugbank_df = pd.read_csv(
        drugbank_file,
        sep="\t"
    )

elif drugbank_file.suffix == ".csv":
    drugbank_df = pd.read_csv(
        drugbank_file
    )

else:
    raise ValueError(
        f"Unsupported DrugBank file format: {drugbank_file.suffix}"
    )

# --------------------------------------------------
# BASIC INSPECTION
# --------------------------------------------------

print("\n" + "=" * 100)
print("DRUGBANK DATASET INSPECTION")
print("=" * 100)

print("\nShape:")
print(drugbank_df.shape)

print("\nColumns:")
print(drugbank_df.columns.tolist())

print("\nData types:")
print(drugbank_df.dtypes)

print("\nFirst 5 rows:")
display(drugbank_df.head())

print("\nMissing values:")
print(drugbank_df.isnull().sum())

print("\nExact duplicate rows:")
print(drugbank_df.duplicated().sum())

print("\nUnique interaction labels:")
if "Y" in drugbank_df.columns:
    print(drugbank_df["Y"].nunique())
    print("\nTop interaction labels:")
    print(drugbank_df["Y"].value_counts().head(20))

DrugBank candidate files:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/tdc/drugbank.tab

Detected DrugBank file:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/tdc/drugbank.tab

DRUGBANK DATASET INSPECTION

Shape:
(191808, 6)

Columns:
['ID1', 'ID2', 'Y', 'Map', 'X1', 'X2']

Data types:
ID1    object
ID2    object
Y       int64
Map    object
X1     object
X2     object
dtype: object

First 5 rows:


,ID1,ID2,Y,Map,X1,X2
0,DB04571,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC1=CC2=CC3=C(OC(=O)C=C3C)C(C)=C2O1,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
1,DB00855,DB00460,1,#Drug1 may increase the photosensitizing activ...,NCC(=O)CCC(O)=O,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
2,DB09536,DB00460,1,#Drug1 may increase the photosensitizing activ...,O=[Ti]=O,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
3,DB01600,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC(C(O)=O)C1=CC=C(S1)C(=O)C1=CC=CC=C1,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
4,DB09000,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC(CN(C)C)CN1C2=CC=CC=C2SC2=C1C=C(C=C2)C#N,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...



Missing values:
ID1    0
ID2    0
Y      0
Map    0
X1     0
X2     0
dtype: int64

Exact duplicate rows:
0

Unique interaction labels:
86

Top interaction labels:
Y
49    60751
47    34360
73    23779
75     9470
60     8397
70     7786
20     6140
16     5413
4      5011
6      3160
37     3089
9      2109
72     1825
54     1277
83     1204
58     1043
32     1011
27      936
67      930
64      803
Name: count, dtype: int64


In [9]:
def choose_twosides_file(files):
    for file in files:
        name = file.name.lower()

        if "twosides" in name:
            return file

    for file in files:
        try:
            sample = pd.read_csv(file, nrows=2)

            columns = set(sample.columns)

            required = {
                "ID1",
                "ID2",
                "Y",
                "Side Effect Name"
            }

            if required.issubset(columns):
                return file

        except Exception:
            pass

    return None


twosides_file = choose_twosides_file(twosides_files)

print("Detected TWOSIDES file:")
print(twosides_file)

Detected TWOSIDES file:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/twosides/twosides.csv


In [13]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# LOCATE DRUGBANK FILE
# --------------------------------------------------

drugbank_candidates = (
    list(TDC_DIR.glob("*.tab")) +
    list(TDC_DIR.glob("*.csv")) +
    list(TDC_DIR.glob("*.tsv"))
)

print("DrugBank candidate files:")
for file in drugbank_candidates:
    print(file)

if not drugbank_candidates:
    raise FileNotFoundError(
        f"No DrugBank dataset file found inside: {TDC_DIR}"
    )

drugbank_file = drugbank_candidates[0]

print("\nDetected DrugBank file:")
print(drugbank_file)


# --------------------------------------------------
# INSPECT RAW FILE
# --------------------------------------------------

print("\n" + "=" * 100)
print("RAW FILE INSPECTION")
print("=" * 100)

with open(drugbank_file, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f"RAW LINE {i+1}:")
        print(repr(f.readline()))
        print()


# --------------------------------------------------
# LOAD DRUGBANK DATA
# --------------------------------------------------

# TDC DrugBank file uses literal escaped tab characters
drugbank_df = pd.read_csv(
    drugbank_file,
    sep=r"\\t",
    engine="python"
)


# --------------------------------------------------
# CLEAN COLUMN NAMES
# --------------------------------------------------

drugbank_df.columns = [
    col.replace("\\t", "").strip()
    for col in drugbank_df.columns
]


# --------------------------------------------------
# BASIC INSPECTION
# --------------------------------------------------

print("\nDrugBank dataset loaded successfully!")

print("\nShape:")
print(drugbank_df.shape)

print("\nColumns:")
print(drugbank_df.columns.tolist())

print("\nData types:")
print(drugbank_df.dtypes)

print("\nMissing values:")
print(drugbank_df.isnull().sum())

print("\nFirst 5 rows:")
display(drugbank_df.head())

DrugBank candidate files:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/tdc/drugbank.tab

Detected DrugBank file:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/tdc/drugbank.tab

RAW FILE INSPECTION
RAW LINE 1:
'ID1\tID2\tY\tMap\tX1\tX2\n'

RAW LINE 2:
'"DB04571"\t"DB00460"\t1\t"#Drug1 may increase the photosensitizing activities of #Drug2."\t"CC1=CC2=CC3=C(OC(=O)C=C3C)C(C)=C2O1"\t"COC(=O)CCC1=C2NC(\\C=C3/N=C(/C=C4\\N\\C(=C/C5=N/C(=C\\2)/C(CCC(O)=O)=C5C)C(C=C)=C4C)C2=CC=C([C@@H](C(=O)OC)[C@@]32C)C(=O)OC)=C1C"\n'

RAW LINE 3:
'"DB00855"\t"DB00460"\t1\t"#Drug1 may increase the photosensitizing activities of #Drug2."\t"NCC(=O)CCC(O)=O"\t"COC(=O)CCC1=C2NC(\\C=C3/N=C(/C=C4\\N\\C(=C/C5=N/C(=C\\2)/C(CCC(O)=O)=C5C)C(C=C)=C4C)C2=CC=C([C@@H](C(=O)OC)[C@@]32C)C(=O)OC)=C1C"\n'


DrugBank dataset loaded successfully!

Shape:
(191808, 1)

Columns:
['ID1\tID2\tY\tMap\tX1\tX2']

Data types:
ID1\tID2\tY\tMap\tX1\tX2    object
dtype: object

Missing values:
ID1\tID

,ID1\tID2\tY\tMap\tX1\tX2
0,"""DB04571""\t""DB00460""\t1\t""#Drug1 may increase ..."
1,"""DB00855""\t""DB00460""\t1\t""#Drug1 may increase ..."
2,"""DB09536""\t""DB00460""\t1\t""#Drug1 may increase ..."
3,"""DB01600""\t""DB00460""\t1\t""#Drug1 may increase ..."
4,"""DB09000""\t""DB00460""\t1\t""#Drug1 may increase ..."


In [14]:
import pandas as pd

# --------------------------------------------------
# FORCE FRESH LOAD OF DRUGBANK
# --------------------------------------------------

print("Reloading DrugBank dataset...")

drugbank_df = pd.read_csv(
    drugbank_file,
    delimiter="\t",
    encoding="utf-8",
    quotechar='"'
)

# --------------------------------------------------
# VERIFY STRUCTURE
# --------------------------------------------------

print("\n" + "=" * 100)
print("DRUGBANK DATASET LOADED")
print("=" * 100)

print("\nShape:")
print(drugbank_df.shape)

print("\nColumns:")
print(drugbank_df.columns.tolist())

print("\nNumber of columns:")
print(len(drugbank_df.columns))

print("\nData types:")
print(drugbank_df.dtypes)

print("\nMissing values:")
print(drugbank_df.isnull().sum())

print("\nFirst 5 rows:")
display(drugbank_df.head())

# --------------------------------------------------
# EXPECTED VALIDATION
# --------------------------------------------------

expected_columns = ["ID1", "ID2", "Y", "Map", "X1", "X2"]

print("\n" + "=" * 100)
print("STRUCTURE VALIDATION")
print("=" * 100)

print("Expected columns:", expected_columns)
print("Actual columns:  ", drugbank_df.columns.tolist())

if list(drugbank_df.columns) == expected_columns:
    print("\nSUCCESS: DrugBank columns loaded correctly!")
else:
    print("\nWARNING: Column structure does not match expected format.")

Reloading DrugBank dataset...

DRUGBANK DATASET LOADED

Shape:
(191808, 6)

Columns:
['ID1', 'ID2', 'Y', 'Map', 'X1', 'X2']

Number of columns:
6

Data types:
ID1    object
ID2    object
Y       int64
Map    object
X1     object
X2     object
dtype: object

Missing values:
ID1    0
ID2    0
Y      0
Map    0
X1     0
X2     0
dtype: int64

First 5 rows:


,ID1,ID2,Y,Map,X1,X2
0,DB04571,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC1=CC2=CC3=C(OC(=O)C=C3C)C(C)=C2O1,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
1,DB00855,DB00460,1,#Drug1 may increase the photosensitizing activ...,NCC(=O)CCC(O)=O,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
2,DB09536,DB00460,1,#Drug1 may increase the photosensitizing activ...,O=[Ti]=O,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
3,DB01600,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC(C(O)=O)C1=CC=C(S1)C(=O)C1=CC=CC=C1,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
4,DB09000,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC(CN(C)C)CN1C2=CC=CC=C2SC2=C1C=C(C=C2)C#N,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...



STRUCTURE VALIDATION
Expected columns: ['ID1', 'ID2', 'Y', 'Map', 'X1', 'X2']
Actual columns:   ['ID1', 'ID2', 'Y', 'Map', 'X1', 'X2']

SUCCESS: DrugBank columns loaded correctly!


In [15]:
# ================================================================
# COMPLETE DRUGBANK RAW DATASET INSPECTION
# ================================================================

print("=" * 100)
print("DRUGBANK COMPLETE DATASET INSPECTION")
print("=" * 100)


# ------------------------------------------------
# 1. BASIC DATASET INFORMATION
# ------------------------------------------------

print("\n1. BASIC INFORMATION")
print("-" * 100)

print("Dataset shape:", drugbank_df.shape)
print("Total records:", len(drugbank_df))
print("Total columns:", len(drugbank_df.columns))

print("\nColumn names:")
print(drugbank_df.columns.tolist())

print("\nData types:")
print(drugbank_df.dtypes)


# ------------------------------------------------
# 2. MISSING VALUES
# ------------------------------------------------

print("\n2. MISSING VALUES")
print("-" * 100)

missing = drugbank_df.isnull().sum()

print(missing)

print("\nTotal missing values in dataset:")
print(missing.sum())


# ------------------------------------------------
# 3. EXACT DUPLICATE ROWS
# ------------------------------------------------

print("\n3. EXACT DUPLICATE ROWS")
print("-" * 100)

exact_duplicates = drugbank_df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)


# ------------------------------------------------
# 4. INTERACTION TYPE ANALYSIS
# ------------------------------------------------

print("\n4. INTERACTION TYPE ANALYSIS")
print("-" * 100)

print("Unique Y interaction labels:")
print(drugbank_df["Y"].nunique())

print("\nAll interaction labels:")
print(sorted(drugbank_df["Y"].unique()))

print("\nInteraction label distribution:")
print(drugbank_df["Y"].value_counts().sort_index())

print("\nMost common interaction types:")
print(drugbank_df["Y"].value_counts().head(20))

print("\nLeast common interaction types:")
print(drugbank_df["Y"].value_counts().tail(20))


# ------------------------------------------------
# 5. DRUG PAIR ANALYSIS
# ------------------------------------------------

print("\n5. DRUG PAIR ANALYSIS")
print("-" * 100)

# Number of unique directed pairs
unique_directed_pairs = drugbank_df[
    ["ID1", "ID2"]
].drop_duplicates()

print("Unique directed drug pairs:", len(unique_directed_pairs))

# Count repeated directed pairs
pair_counts = drugbank_df.groupby(
    ["ID1", "ID2"]
).size()

repeated_pairs = pair_counts[pair_counts > 1]

print("Directed pairs with more than one record:",
      len(repeated_pairs))

print("Rows belonging to repeated directed pairs:",
      pair_counts[pair_counts > 1].sum())

print("\nMaximum number of interaction records for one directed pair:")
print(pair_counts.max())

print("\nAverage interaction records per directed pair:")
print(pair_counts.mean())


# ------------------------------------------------
# 6. PAIRS WITH MULTIPLE INTERACTION TYPES
# ------------------------------------------------

print("\n6. MULTI-RELATIONSHIP PAIRS")
print("-" * 100)

pair_label_counts = drugbank_df.groupby(
    ["ID1", "ID2"]
)["Y"].nunique()

multi_type_pairs = pair_label_counts[
    pair_label_counts > 1
]

print("Drug pairs with multiple interaction types:",
      len(multi_type_pairs))

print("\nDistribution of number of interaction types per pair:")
print(pair_label_counts.value_counts().sort_index())

print("\nExamples of multi-interaction drug pairs:")

multi_pair_examples = (
    drugbank_df[
        drugbank_df.set_index(
            ["ID1", "ID2"]
        ).index.isin(multi_type_pairs.index)
    ]
    .groupby(["ID1", "ID2"])["Y"]
    .agg(lambda x: sorted(x.unique()))
    .head(20)
)

print(multi_pair_examples)


# ------------------------------------------------
# 7. CHECK FOR DUPLICATE PAIR + LABEL RECORDS
# ------------------------------------------------

print("\n7. DUPLICATE DRUG PAIR + INTERACTION LABEL RECORDS")
print("-" * 100)

duplicate_pair_label = drugbank_df.duplicated(
    subset=["ID1", "ID2", "Y"]
).sum()

print(
    "Duplicate ID1 + ID2 + Y records:",
    duplicate_pair_label
)


# ------------------------------------------------
# 8. UNIQUE DRUG ANALYSIS
# ------------------------------------------------

print("\n8. UNIQUE DRUG ANALYSIS")
print("-" * 100)

all_drugs = set(
    drugbank_df["ID1"]
).union(
    set(drugbank_df["ID2"])
)

print("Total unique drugs:", len(all_drugs))

print("\nUnique drugs appearing as Drug1:", 
      drugbank_df["ID1"].nunique())

print("Unique drugs appearing as Drug2:", 
      drugbank_df["ID2"].nunique())


# ------------------------------------------------
# 9. STRUCTURE CONSISTENCY
# ------------------------------------------------

print("\n9. CHEMICAL STRUCTURE CONSISTENCY")
print("-" * 100)

drug1_structure_counts = drugbank_df.groupby(
    "ID1"
)["X1"].nunique()

drug2_structure_counts = drugbank_df.groupby(
    "ID2"
)["X2"].nunique()

print(
    "Drug1 IDs having multiple structures:",
    (drug1_structure_counts > 1).sum()
)

print(
    "Drug2 IDs having multiple structures:",
    (drug2_structure_counts > 1).sum()
)


# ------------------------------------------------
# 10. REVERSE PAIR ANALYSIS
# ------------------------------------------------

print("\n10. REVERSE PAIR ANALYSIS")
print("-" * 100)

directed_pair_set = set(
    zip(
        drugbank_df["ID1"],
        drugbank_df["ID2"]
    )
)

reverse_counterparts = []

for drug1, drug2 in directed_pair_set:
    if (drug2, drug1) in directed_pair_set:
        reverse_counterparts.append(
            (drug1, drug2)
        )

print(
    "Directed pairs having a reverse counterpart:",
    len(reverse_counterparts)
)

print(
    "Total unique directed pairs:",
    len(directed_pair_set)
)


# ------------------------------------------------
# 11. INTERACTION DESCRIPTION ANALYSIS
# ------------------------------------------------

print("\n11. INTERACTION DESCRIPTION ANALYSIS")
print("-" * 100)

print(
    "Unique interaction descriptions:",
    drugbank_df["Map"].nunique()
)

print(
    "\nInteraction labels mapping to multiple descriptions:"
)

label_to_descriptions = drugbank_df.groupby(
    "Y"
)["Map"].nunique()

print(
    label_to_descriptions[
        label_to_descriptions > 1
    ]
)


print(
    "\nDescriptions mapping to multiple labels:"
)

description_to_labels = drugbank_df.groupby(
    "Map"
)["Y"].nunique()

print(
    description_to_labels[
        description_to_labels > 1
    ]
)


# ------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 100)
print("FINAL DRUGBANK SUMMARY")
print("=" * 100)

print(f"""
Total records: {len(drugbank_df):,}
Columns: {len(drugbank_df.columns)}
Unique drugs: {len(all_drugs):,}
Unique directed drug pairs: {len(unique_directed_pairs):,}
Interaction types: {drugbank_df["Y"].nunique()}
Exact duplicate rows: {exact_duplicates:,}
Duplicate pair + label records: {duplicate_pair_label:,}
Pairs with multiple interaction types: {len(multi_type_pairs):,}
Maximum interaction records for one pair: {pair_counts.max()}
Reverse directed pair records: {len(reverse_counterparts):,}
Missing values: {missing.sum()}
""")

DRUGBANK COMPLETE DATASET INSPECTION

1. BASIC INFORMATION
----------------------------------------------------------------------------------------------------
Dataset shape: (191808, 6)
Total records: 191808
Total columns: 6

Column names:
['ID1', 'ID2', 'Y', 'Map', 'X1', 'X2']

Data types:
ID1    object
ID2    object
Y       int64
Map    object
X1     object
X2     object
dtype: object

2. MISSING VALUES
----------------------------------------------------------------------------------------------------
ID1    0
ID2    0
Y      0
Map    0
X1     0
X2     0
dtype: int64

Total missing values in dataset:
0

3. EXACT DUPLICATE ROWS
----------------------------------------------------------------------------------------------------
Exact duplicate rows: 0

4. INTERACTION TYPE ANALYSIS
----------------------------------------------------------------------------------------------------
Unique Y interaction labels:
86

All interaction labels:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 

In [16]:
# ================================================================
# RXNAV / RXNORM API DISCOVERY AND INITIAL INSPECTION
# ================================================================

import requests
import pandas as pd
import json
from pathlib import Path
from datetime import datetime


# ------------------------------------------------
# 1. API CONFIGURATION
# ------------------------------------------------

RXNAV_BASE_URL = "https://rxnav.nlm.nih.gov/REST"

print("=" * 100)
print("RXNAV / RXNORM API DISCOVERY")
print("=" * 100)

print("\nBase URL:")
print(RXNAV_BASE_URL)


# ------------------------------------------------
# 2. TEST API CONNECTIVITY
# ------------------------------------------------

print("\n2. API CONNECTIVITY TEST")
print("-" * 100)

test_drug = "Aspirin"

response = requests.get(
    f"{RXNAV_BASE_URL}/rxcui.json",
    params={"name": test_drug},
    timeout=30
)

print("Test drug:", test_drug)
print("Status code:", response.status_code)

if response.status_code == 200:
    print("SUCCESS: RxNav API is reachable!")
    print("Response:")
    print(response.json())
else:
    print("WARNING: API request failed")


# ------------------------------------------------
# 3. HELPER FUNCTION: GET RxCUI FROM DRUG NAME
# ------------------------------------------------

def get_rxcui_from_name(drug_name):
    """
    Search RxNorm for a drug name
    and return available RxCUI values.
    """

    try:

        response = requests.get(
            f"{RXNAV_BASE_URL}/rxcui.json",
            params={"name": drug_name},
            timeout=30
        )

        if response.status_code != 200:
            return {
                "query": drug_name,
                "status": "request_failed",
                "rxcuis": []
            }

        data = response.json()

        id_group = data.get("idGroup", {})

        rxcuis = id_group.get(
            "rxnormId",
            []
        )

        return {
            "query": drug_name,
            "status": "success",
            "rxcuis": rxcuis
        }

    except Exception as e:

        return {
            "query": drug_name,
            "status": "error",
            "error": str(e),
            "rxcuis": []
        }


# ------------------------------------------------
# 4. TEST MULTIPLE DRUG NAME LOOKUPS
# ------------------------------------------------

print("\n3. MULTIPLE DRUG NAME LOOKUPS")
print("-" * 100)

test_drugs = [
    "Aspirin",
    "Ibuprofen",
    "Paracetamol",
    "Metformin",
    "Amoxicillin",
    "Warfarin"
]

lookup_results = []

for drug in test_drugs:

    result = get_rxcui_from_name(drug)

    lookup_results.append(result)

    print("\nDrug:", drug)
    print("Status:", result["status"])
    print("RxCUIs:", result["rxcuis"])


# ------------------------------------------------
# 5. CREATE LOOKUP DATAFRAME
# ------------------------------------------------

print("\n4. LOOKUP SUMMARY")
print("-" * 100)

lookup_df = pd.DataFrame(
    [
        {
            "drug_name": result["query"],
            "status": result["status"],
            "rxcui_count": len(result["rxcuis"]),
            "rxcuis": result["rxcuis"]
        }
        for result in lookup_results
    ]
)

display(lookup_df)


# ------------------------------------------------
# 6. HELPER FUNCTION: GET RXNORM PROPERTIES
# ------------------------------------------------

def get_rxcui_properties(rxcui):
    """
    Retrieve basic RxNorm concept properties.
    """

    try:

        response = requests.get(
            f"{RXNAV_BASE_URL}/rxcui/{rxcui}/properties.json",
            timeout=30
        )

        if response.status_code != 200:
            return {
                "rxcui": rxcui,
                "status": "request_failed"
            }

        data = response.json()

        properties = data.get(
            "properties",
            {}
        )

        return {
            "rxcui": rxcui,
            "status": "success",
            "properties": properties
        }

    except Exception as e:

        return {
            "rxcui": rxcui,
            "status": "error",
            "error": str(e)
        }


# ------------------------------------------------
# 7. INSPECT ONE REAL RxCUI
# ------------------------------------------------

print("\n5. RxCUI PROPERTY INSPECTION")
print("-" * 100)

first_valid_rxcui = None

for result in lookup_results:

    if len(result["rxcuis"]) > 0:

        first_valid_rxcui = result["rxcuis"][0]
        break


if first_valid_rxcui:

    print("Selected RxCUI:", first_valid_rxcui)

    properties_result = get_rxcui_properties(
        first_valid_rxcui
    )

    print("\nProperties response:")

    print(
        json.dumps(
            properties_result,
            indent=4
        )
    )

else:

    print("No valid RxCUI found for property inspection.")


# ------------------------------------------------
# 8. SAVE TEST RESULTS LOCALLY
# ------------------------------------------------

print("\n6. SAVING RAW RXNAV RESULTS")
print("-" * 100)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = (
    RXNAV_DIR /
    f"rxnav_initial_lookup_{timestamp}.json"
)

output_data = {
    "created_at": datetime.now().isoformat(),
    "test_drugs": test_drugs,
    "lookup_results": lookup_results
}

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        indent=4
    )

print("Saved RxNav test results to:")
print(output_file)


# ------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 100)
print("RXNAV INITIAL DISCOVERY SUMMARY")
print("=" * 100)

successful_lookups = (
    lookup_df["rxcui_count"] > 0
).sum()

print(f"""
Drugs tested: {len(test_drugs)}
Successful RxNorm matches: {successful_lookups}
Failed/no-match lookups: {len(test_drugs) - successful_lookups}
Raw API responses saved: Yes
RxNav directory: {RXNAV_DIR}
""")

RXNAV / RXNORM API DISCOVERY

Base URL:
https://rxnav.nlm.nih.gov/REST

2. API CONNECTIVITY TEST
----------------------------------------------------------------------------------------------------
Test drug: Aspirin
Status code: 200
SUCCESS: RxNav API is reachable!
Response:
{'idGroup': {'rxnormId': ['1191']}}

3. MULTIPLE DRUG NAME LOOKUPS
----------------------------------------------------------------------------------------------------

Drug: Aspirin
Status: success
RxCUIs: ['1191']

Drug: Ibuprofen
Status: success
RxCUIs: ['5640']

Drug: Paracetamol
Status: success
RxCUIs: ['161']

Drug: Metformin
Status: success
RxCUIs: ['6809']

Drug: Amoxicillin
Status: success
RxCUIs: ['723']

Drug: Warfarin
Status: success
RxCUIs: ['11289']

4. LOOKUP SUMMARY
----------------------------------------------------------------------------------------------------


,drug_name,status,rxcui_count,rxcuis
0,Aspirin,success,1,[1191]
1,Ibuprofen,success,1,[5640]
2,Paracetamol,success,1,[161]
3,Metformin,success,1,[6809]
4,Amoxicillin,success,1,[723]
5,Warfarin,success,1,[11289]



5. RxCUI PROPERTY INSPECTION
----------------------------------------------------------------------------------------------------
Selected RxCUI: 1191

Properties response:
{
    "rxcui": "1191",
    "status": "success",
    "properties": {
        "rxcui": "1191",
        "name": "aspirin",
        "synonym": "",
        "tty": "IN",
        "language": "ENG",
        "suppress": "N",
        "umlscui": ""
    }
}

6. SAVING RAW RXNAV RESULTS
----------------------------------------------------------------------------------------------------
Saved RxNav test results to:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/rxnav/rxnav_initial_lookup_20260828_153827.json

RXNAV INITIAL DISCOVERY SUMMARY

Drugs tested: 6
Successful RxNorm matches: 6
Failed/no-match lookups: 0
Raw API responses saved: Yes
RxNav directory: /Users/apple/Documents/SIH2026/Prescription_safety_graph/data/raw/rxnav



In [18]:
# ================================================================
# IDENTIFIER DISCOVERY AND CROSS-DATASET INSPECTION
# ================================================================

import pandas as pd
from pathlib import Path


print("=" * 100)
print("IDENTIFIER DISCOVERY AND CROSS-DATASET INSPECTION")
print("=" * 100)


# ================================================================
# 1. DRUGBANK IDENTIFIER INSPECTION
# ================================================================

print("\n1. DRUGBANK IDENTIFIERS")
print("-" * 100)

drugbank_ids = pd.concat(
    [
        drugbank_df["ID1"],
        drugbank_df["ID2"]
    ]
).dropna().astype(str).unique()

print("Unique DrugBank drug identifiers:", len(drugbank_ids))

print("\nFirst 20 DrugBank identifiers:")
print(sorted(drugbank_ids)[:20])

print("\nLast 20 DrugBank identifiers:")
print(sorted(drugbank_ids)[-20:])

print("\nIdentifier examples:")
for drug_id in sorted(drugbank_ids)[:10]:
    print(drug_id)


# ================================================================
# 2. TWOSIDES IDENTIFIER INSPECTION
# ================================================================

print("\n" + "=" * 100)
print("2. TWOSIDES IDENTIFIERS")
print("-" * 100)

twosides_ids = pd.concat(
    [
        twosides_df["ID1"],
        twosides_df["ID2"]
    ]
).dropna().astype(str).unique()

print("Unique TWOSIDES drug identifiers:", len(twosides_ids))

print("\nFirst 20 TWOSIDES identifiers:")
print(sorted(twosides_ids)[:20])

print("\nLast 20 TWOSIDES identifiers:")
print(sorted(twosides_ids)[-20:])

print("\nIdentifier examples:")
for drug_id in sorted(twosides_ids)[:10]:
    print(drug_id)


# ================================================================
# 3. IDENTIFIER FORMAT ANALYSIS
# ================================================================

print("\n" + "=" * 100)
print("3. IDENTIFIER FORMAT ANALYSIS")
print("-" * 100)

drugbank_prefix_counts = (
    pd.Series(drugbank_ids)
    .str.extract(r"^([A-Za-z]+)", expand=False)
    .value_counts()
)

twosides_prefix_counts = (
    pd.Series(twosides_ids)
    .str.extract(r"^([A-Za-z]+)", expand=False)
    .value_counts()
)

print("\nDrugBank identifier prefixes:")
print(drugbank_prefix_counts)

print("\nTWOSIDES identifier prefixes:")
print(twosides_prefix_counts)


# ================================================================
# 4. DIRECT IDENTIFIER OVERLAP
# ================================================================

print("\n" + "=" * 100)
print("4. DIRECT IDENTIFIER OVERLAP")
print("-" * 100)

drugbank_id_set = set(drugbank_ids)
twosides_id_set = set(twosides_ids)

direct_overlap = (
    drugbank_id_set
    .intersection(twosides_id_set)
)

print("DrugBank unique IDs:", len(drugbank_id_set))
print("TWOSIDES unique IDs:", len(twosides_id_set))
print("Directly identical identifiers:", len(direct_overlap))

if len(direct_overlap) > 0:
    print("\nOverlapping identifiers:")
    print(sorted(direct_overlap)[:50])

else:
    print("\nNo direct identifier overlap found.")

    print(
        "\nThis confirms that DrugBank IDs and "
        "TWOSIDES identifiers belong to different identifier systems."
    )


# ================================================================
# 5. CHEMICAL STRUCTURE AVAILABILITY
# ================================================================

print("\n" + "=" * 100)
print("5. CHEMICAL STRUCTURE AVAILABILITY")
print("-" * 100)

drugbank_structure_map = pd.concat(
    [
        drugbank_df[
            ["ID1", "X1"]
        ].rename(
            columns={
                "ID1": "drug_id",
                "X1": "structure"
            }
        ),

        drugbank_df[
            ["ID2", "X2"]
        ].rename(
            columns={
                "ID2": "drug_id",
                "X2": "structure"
            }
        )
    ]
)

twosides_structure_map = pd.concat(
    [
        twosides_df[
            ["ID1", "X1"]
        ].rename(
            columns={
                "ID1": "drug_id",
                "X1": "structure"
            }
        ),

        twosides_df[
            ["ID2", "X2"]
        ].rename(
            columns={
                "ID2": "drug_id",
                "X2": "structure"
            }
        )
    ]
)


# Remove duplicates

drugbank_structure_map = (
    drugbank_structure_map
    .drop_duplicates()
)

twosides_structure_map = (
    twosides_structure_map
    .drop_duplicates()
)


print("\nDrugBank identifier → structure records:")
print(len(drugbank_structure_map))

print("DrugBank unique drugs with structures:")
print(
    drugbank_structure_map[
        "drug_id"
    ].nunique()
)

print("\nTWOSIDES identifier → structure records:")
print(len(twosides_structure_map))

print("TWOSIDES unique drugs with structures:")
print(
    twosides_structure_map[
        "drug_id"
    ].nunique()
)


# ================================================================
# 6. STRUCTURE CONSISTENCY CHECK
# ================================================================

print("\n" + "=" * 100)
print("6. STRUCTURE CONSISTENCY CHECK")
print("-" * 100)

drugbank_structure_counts = (
    drugbank_structure_map
    .groupby("drug_id")["structure"]
    .nunique()
)

twosides_structure_counts = (
    twosides_structure_map
    .groupby("drug_id")["structure"]
    .nunique()
)

drugbank_multiple_structures = (
    drugbank_structure_counts > 1
).sum()

twosides_multiple_structures = (
    twosides_structure_counts > 1
).sum()

print(
    "DrugBank IDs with multiple structures:",
    drugbank_multiple_structures
)

print(
    "TWOSIDES IDs with multiple structures:",
    twosides_multiple_structures
)


# ================================================================
# 7. EXACT STRUCTURE OVERLAP
# ================================================================

print("\n" + "=" * 100)
print("7. EXACT STRUCTURE OVERLAP BETWEEN DATASETS")
print("-" * 100)


drugbank_unique_structure_map = (
    drugbank_structure_map
    .drop_duplicates(
        subset=["drug_id"]
    )
)

twosides_unique_structure_map = (
    twosides_structure_map
    .drop_duplicates(
        subset=["drug_id"]
    )
)


exact_structure_matches = (
    drugbank_unique_structure_map
    .merge(
        twosides_unique_structure_map,
        on="structure",
        how="inner",
        suffixes=(
            "_drugbank",
            "_twosides"
        )
    )
)


print(
    "Exact structure matches:",
    len(exact_structure_matches)
)

print("\nFirst 20 exact structure matches:")

display(
    exact_structure_matches[
        [
            "drug_id_drugbank",
            "drug_id_twosides",
            "structure"
        ]
    ].head(20)
)


# ================================================================
# 8. ONE-TO-ONE STRUCTURE MAPPING CHECK
# ================================================================

print("\n" + "=" * 100)
print("8. ONE-TO-ONE STRUCTURE MAPPING CHECK")
print("-" * 100)


drugbank_ids_per_structure = (
    exact_structure_matches
    .groupby("structure")
    ["drug_id_drugbank"]
    .nunique()
)

twosides_ids_per_structure = (
    exact_structure_matches
    .groupby("structure")
    ["drug_id_twosides"]
    .nunique()
)


ambiguous_structure_matches = (
    (drugbank_ids_per_structure > 1) |
    (twosides_ids_per_structure > 1)
).sum()


print(
    "Ambiguous exact-structure mappings:",
    ambiguous_structure_matches
)


one_to_one_matches = exact_structure_matches[
    exact_structure_matches[
        "structure"
    ].map(
        drugbank_ids_per_structure
    ).eq(1)
    &
    exact_structure_matches[
        "structure"
    ].map(
        twosides_ids_per_structure
    ).eq(1)
]


print(
    "One-to-one exact structure matches:",
    len(one_to_one_matches)
)


# ================================================================
# 9. OVERLAP COVERAGE
# ================================================================

print("\n" + "=" * 100)
print("9. CROSS-DATASET OVERLAP COVERAGE")
print("-" * 100)

drugbank_matched_ids = (
    one_to_one_matches[
        "drug_id_drugbank"
    ].nunique()
)

twosides_matched_ids = (
    one_to_one_matches[
        "drug_id_twosides"
    ].nunique()
)


drugbank_coverage = (
    drugbank_matched_ids
    /
    len(drugbank_id_set)
    *
    100
)

twosides_coverage = (
    twosides_matched_ids
    /
    len(twosides_id_set)
    *
    100
)


print(
    f"DrugBank drugs matched through structure: "
    f"{drugbank_matched_ids} / {len(drugbank_id_set)} "
    f"({drugbank_coverage:.2f}%)"
)

print(
    f"TWOSIDES drugs matched through structure: "
    f"{twosides_matched_ids} / {len(twosides_id_set)} "
    f"({twosides_coverage:.2f}%)"
)


# ================================================================
# 10. CREATE INITIAL CROSSWALK
# ================================================================

print("\n" + "=" * 100)
print("10. INITIAL STRUCTURE-BASED CROSSWALK")
print("-" * 100)

initial_crosswalk = (
    one_to_one_matches[
        [
            "drug_id_drugbank",
            "drug_id_twosides",
            "structure"
        ]
    ]
    .rename(
        columns={
            "drug_id_drugbank": "drugbank_id",
            "drug_id_twosides": "twosides_id"
        }
    )
    .drop_duplicates()
)


print(
    "Initial one-to-one crosswalk records:",
    len(initial_crosswalk)
)

print("\nFirst 20 crosswalk records:")

display(
    initial_crosswalk.head(20)
)


# ================================================================
# 11. SAVE DISCOVERY RESULTS
# ================================================================

print("\n" + "=" * 100)
print("11. SAVING DISCOVERY RESULTS")
print("-" * 100)


crosswalk_output = (
    CROSSWALKS_DIR /
    "drugbank_twosides_structure_crosswalk.csv"
)


initial_crosswalk.to_csv(
    crosswalk_output,
    index=False
)


print("Saved initial crosswalk:")
print(crosswalk_output)


# ================================================================
# 12. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 100)
print("IDENTIFIER DISCOVERY SUMMARY")
print("=" * 100)

print(f"""
DrugBank identifier system:
Unique IDs: {len(drugbank_id_set)}

TWOSIDES identifier system:
Unique IDs: {len(twosides_id_set)}

Direct identifier overlap:
{len(direct_overlap)}

Exact structure matches:
{len(exact_structure_matches)}

One-to-one structure matches:
{len(one_to_one_matches)}

DrugBank coverage through structure:
{drugbank_coverage:.2f}%

TWOSIDES coverage through structure:
{twosides_coverage:.2f}%

Crosswalk saved:
{crosswalk_output}
""")

IDENTIFIER DISCOVERY AND CROSS-DATASET INSPECTION

1. DRUGBANK IDENTIFIERS
----------------------------------------------------------------------------------------------------
Unique DrugBank drug identifiers: 1706

First 20 DrugBank identifiers:
['DB00006', 'DB00014', 'DB00027', 'DB00035', 'DB00080', 'DB00091', 'DB00104', 'DB00115', 'DB00120', 'DB00122', 'DB00126', 'DB00130', 'DB00134', 'DB00136', 'DB00140', 'DB00142', 'DB00146', 'DB00150', 'DB00152', 'DB00153']

Last 20 DrugBank identifiers:
['DB13139', 'DB13142', 'DB13145', 'DB13146', 'DB13153', 'DB13158', 'DB13167', 'DB13179', 'DB13231', 'DB13323', 'DB13450', 'DB13595', 'DB13747', 'DB13783', 'DB13872', 'DB13873', 'DB13874', 'DB13878', 'DB13879', 'DB13925']

Identifier examples:
DB00006
DB00014
DB00027
DB00035
DB00080
DB00091
DB00104
DB00115
DB00120
DB00122

2. TWOSIDES IDENTIFIERS
----------------------------------------------------------------------------------------------------


NameError: name 'twosides_df' is not defined

In [19]:
# ====================================================================================
# IDENTIFIER DISCOVERY AND CROSS-DATASET INSPECTION
# ====================================================================================

import pandas as pd

print("=" * 100)
print("IDENTIFIER DISCOVERY AND CROSS-DATASET INSPECTION")
print("=" * 100)


# ====================================================================================
# 1. MAKE SURE DRUGBANK DATAFRAME EXISTS
# ====================================================================================

print("\n1. DRUGBANK IDENTIFIERS")
print("-" * 100)

# Reload from file if the variable is missing
try:
    drugbank_df
    print("Using existing DrugBank dataframe.")
except NameError:
    print("DrugBank dataframe not found in memory. Reloading...")
    drugbank_df = pd.read_csv(
        drugbank_file,
        sep="\t",
        quotechar='"'
    )

drugbank_ids = pd.concat(
    [
        drugbank_df["ID1"],
        drugbank_df["ID2"]
    ]
).dropna().astype(str).unique()

drugbank_ids = sorted(drugbank_ids)

print("Unique DrugBank drug identifiers:", len(drugbank_ids))

print("\nFirst 20 DrugBank identifiers:")
print(drugbank_ids[:20])

print("\nLast 20 DrugBank identifiers:")
print(drugbank_ids[-20:])

print("\nIdentifier examples:")
for identifier in drugbank_ids[:10]:
    print(identifier)


# ====================================================================================
# 2. MAKE SURE TWOSIDES DATAFRAME EXISTS
# ====================================================================================

print("\n" + "=" * 100)
print("2. TWOSIDES IDENTIFIERS")
print("-" * 100)

# Reload TWOSIDES from file if the variable is missing
try:
    twosides_df
    print("Using existing TWOSIDES dataframe.")
except NameError:
    print("TWOSIDES dataframe not found in memory. Reloading from CSV...")
    twosides_df = pd.read_csv(twosides_file)

twosides_ids = pd.concat(
    [
        twosides_df["ID1"],
        twosides_df["ID2"]
    ]
).dropna().astype(str).unique()

twosides_ids = sorted(twosides_ids)

print("Unique TWOSIDES drug identifiers:", len(twosides_ids))

print("\nFirst 20 TWOSIDES identifiers:")
print(twosides_ids[:20])

print("\nLast 20 TWOSIDES identifiers:")
print(twosides_ids[-20:])

print("\nIdentifier examples:")
for identifier in twosides_ids[:10]:
    print(identifier)


# ====================================================================================
# 3. IDENTIFIER FORMAT INSPECTION
# ====================================================================================

print("\n" + "=" * 100)
print("3. IDENTIFIER FORMAT COMPARISON")
print("-" * 100)

print("\nDrugBank format examples:")
for identifier in drugbank_ids[:5]:
    print(identifier)

print("\nTWOSIDES format examples:")
for identifier in twosides_ids[:5]:
    print(identifier)


# ====================================================================================
# 4. DIRECT IDENTIFIER OVERLAP
# ====================================================================================

print("\n" + "=" * 100)
print("4. DIRECT IDENTIFIER OVERLAP")
print("-" * 100)

drugbank_set = set(drugbank_ids)
twosides_set = set(twosides_ids)

direct_overlap = drugbank_set.intersection(twosides_set)

print("DrugBank unique IDs:", len(drugbank_set))
print("TWOSIDES unique IDs:", len(twosides_set))
print("Direct identifier overlap:", len(direct_overlap))

if len(direct_overlap) > 0:
    print("\nExample overlapping identifiers:")
    print(sorted(list(direct_overlap))[:20])
else:
    print("\nNo direct identifier overlap detected.")
    print("This means a crosswalk/mapping layer will be required.")


# ====================================================================================
# 5. IDENTIFIER TYPE CLASSIFICATION
# ====================================================================================

print("\n" + "=" * 100)
print("5. IDENTIFIER TYPE CLASSIFICATION")
print("-" * 100)

drugbank_prefix_counts = pd.Series(
    [x[:2] for x in drugbank_ids]
).value_counts()

twosides_prefix_counts = pd.Series(
    [x.split("000")[0] if "000" in x else x[:3] for x in twosides_ids]
).value_counts()

print("\nDrugBank prefix distribution:")
print(drugbank_prefix_counts.head(10))

print("\nTWOSIDES identifier examples confirm their format:")
print(twosides_ids[:10])


# ====================================================================================
# 6. STRUCTURE-LEVEL CROSS-DATASET POSSIBILITY
# ====================================================================================

print("\n" + "=" * 100)
print("6. STRUCTURE-LEVEL CROSS-DATASET INSPECTION")
print("-" * 100)

# DrugBank structures
drugbank_structures = pd.concat(
    [
        drugbank_df[["ID1", "X1"]].rename(
            columns={"ID1": "drug_id", "X1": "structure"}
        ),
        drugbank_df[["ID2", "X2"]].rename(
            columns={"ID2": "drug_id", "X2": "structure"}
        )
    ]
).drop_duplicates()

print("DrugBank unique ID-structure records:", len(drugbank_structures))
print("DrugBank unique structures:", drugbank_structures["structure"].nunique())


# TWOSIDES structures
twosides_structures = pd.concat(
    [
        twosides_df[["ID1", "X1"]].rename(
            columns={"ID1": "drug_id", "X1": "structure"}
        ),
        twosides_df[["ID2", "X2"]].rename(
            columns={"ID2": "drug_id", "X2": "structure"}
        )
    ]
).drop_duplicates()

print("TWOSIDES unique ID-structure records:", len(twosides_structures))
print("TWOSIDES unique structures:", twosides_structures["structure"].nunique())


# ====================================================================================
# 7. EXACT STRUCTURE OVERLAP
# ====================================================================================

print("\n" + "=" * 100)
print("7. EXACT MOLECULAR STRUCTURE OVERLAP")
print("-" * 100)

drugbank_structure_set = set(
    drugbank_structures["structure"].dropna().astype(str)
)

twosides_structure_set = set(
    twosides_structures["structure"].dropna().astype(str)
)

shared_structures = drugbank_structure_set.intersection(
    twosides_structure_set
)

print("Unique DrugBank structures:", len(drugbank_structure_set))
print("Unique TWOSIDES structures:", len(twosides_structure_set))
print("Exact shared molecular structures:", len(shared_structures))


# ====================================================================================
# 8. CREATE PRELIMINARY STRUCTURE CROSSWALK
# ====================================================================================

print("\n" + "=" * 100)
print("8. PRELIMINARY STRUCTURE-BASED CROSSWALK")
print("-" * 100)

structure_crosswalk = drugbank_structures.merge(
    twosides_structures,
    on="structure",
    how="inner",
    suffixes=("_drugbank", "_twosides")
)

print("Raw structure-based mapping records:", len(structure_crosswalk))

print("\nFirst 20 mappings:")
display(structure_crosswalk.head(20))


# ====================================================================================
# 9. CROSSWALK QUALITY CHECK
# ====================================================================================

print("\n" + "=" * 100)
print("9. PRELIMINARY CROSSWALK QUALITY")
print("-" * 100)

if len(structure_crosswalk) > 0:

    drugbank_to_twosides = (
        structure_crosswalk
        .groupby("drug_id_drugbank")["drug_id_twosides"]
        .nunique()
    )

    twosides_to_drugbank = (
        structure_crosswalk
        .groupby("drug_id_twosides")["drug_id_drugbank"]
        .nunique()
    )

    print(
        "DrugBank IDs participating in structure-based mapping:",
        structure_crosswalk["drug_id_drugbank"].nunique()
    )

    print(
        "TWOSIDES IDs participating in structure-based mapping:",
        structure_crosswalk["drug_id_twosides"].nunique()
    )

    print(
        "DrugBank IDs mapping to exactly one TWOSIDES ID:",
        (drugbank_to_twosides == 1).sum()
    )

    print(
        "DrugBank IDs mapping to multiple TWOSIDES IDs:",
        (drugbank_to_twosides > 1).sum()
    )

    print(
        "TWOSIDES IDs mapping to exactly one DrugBank ID:",
        (twosides_to_drugbank == 1).sum()
    )

    print(
        "TWOSIDES IDs mapping to multiple DrugBank IDs:",
        (twosides_to_drugbank > 1).sum()
    )

else:
    print("No exact structure-based mappings were found.")


# ====================================================================================
# 10. FINAL SUMMARY
# ====================================================================================

print("\n" + "=" * 100)
print("IDENTIFIER DISCOVERY SUMMARY")
print("=" * 100)

print(f"DrugBank unique drugs: {len(drugbank_ids)}")
print(f"TWOSIDES unique drugs: {len(twosides_ids)}")
print(f"Direct identifier overlap: {len(direct_overlap)}")
print(f"DrugBank unique structures: {len(drugbank_structure_set)}")
print(f"TWOSIDES unique structures: {len(twosides_structure_set)}")
print(f"Exact shared structures: {len(shared_structures)}")
print(f"Raw structure-based mappings: {len(structure_crosswalk)}")

print("\nIDENTIFIER TYPES:")
print("DrugBank:", drugbank_ids[0])
print("TWOSIDES:", twosides_ids[0])

print("\nNEXT INTERPRETATION:")
print(
    "Direct IDs are expected to differ because DrugBank uses DB identifiers "
    "while TWOSIDES uses PubChem CID identifiers."
)

print(
    "The exact structure overlap and structure-based crosswalk will tell us "
    "how much of the two datasets can be connected immediately."
)

print("=" * 100)

IDENTIFIER DISCOVERY AND CROSS-DATASET INSPECTION

1. DRUGBANK IDENTIFIERS
----------------------------------------------------------------------------------------------------
Using existing DrugBank dataframe.
Unique DrugBank drug identifiers: 1706

First 20 DrugBank identifiers:
['DB00006', 'DB00014', 'DB00027', 'DB00035', 'DB00080', 'DB00091', 'DB00104', 'DB00115', 'DB00120', 'DB00122', 'DB00126', 'DB00130', 'DB00134', 'DB00136', 'DB00140', 'DB00142', 'DB00146', 'DB00150', 'DB00152', 'DB00153']

Last 20 DrugBank identifiers:
['DB13139', 'DB13142', 'DB13145', 'DB13146', 'DB13153', 'DB13158', 'DB13167', 'DB13179', 'DB13231', 'DB13323', 'DB13450', 'DB13595', 'DB13747', 'DB13783', 'DB13872', 'DB13873', 'DB13874', 'DB13878', 'DB13879', 'DB13925']

Identifier examples:
DB00006
DB00014
DB00027
DB00035
DB00080
DB00091
DB00104
DB00115
DB00120
DB00122

2. TWOSIDES IDENTIFIERS
----------------------------------------------------------------------------------------------------
TWOSIDES datafram

,drug_id_drugbank,structure,drug_id_twosides
0,DB00268,CCCN(CCC)CCC1=C2CC(=O)NC2=CC=C1,CID000005095
1,DB00763,CN1C=CNC1=S,CID001349907
2,DB00435,[N]=O,CID000145068
3,DB00822,CCN(CC)C(=S)SSC(=S)N(CC)CC,CID000003117
4,DB00550,CCCC1=CC(=O)NC(=S)N1,CID000657298
5,DB09153,[Na+].[Cl-],CID000005234
6,DB00761,[Cl-].[K+],CID000004873



9. PRELIMINARY CROSSWALK QUALITY
----------------------------------------------------------------------------------------------------
DrugBank IDs participating in structure-based mapping: 7
TWOSIDES IDs participating in structure-based mapping: 7
DrugBank IDs mapping to exactly one TWOSIDES ID: 7
DrugBank IDs mapping to multiple TWOSIDES IDs: 0
TWOSIDES IDs mapping to exactly one DrugBank ID: 7
TWOSIDES IDs mapping to multiple DrugBank IDs: 0

IDENTIFIER DISCOVERY SUMMARY
DrugBank unique drugs: 1706
TWOSIDES unique drugs: 645
Direct identifier overlap: 0
DrugBank unique structures: 1706
TWOSIDES unique structures: 645
Exact shared structures: 7
Raw structure-based mappings: 7

IDENTIFIER TYPES:
DrugBank: DB00006
TWOSIDES: CID000000085

NEXT INTERPRETATION:
Direct IDs are expected to differ because DrugBank uses DB identifiers while TWOSIDES uses PubChem CID identifiers.
The exact structure overlap and structure-based crosswalk will tell us how much of the two datasets can be connecte

In [20]:
# ====================================================================================
# STEP 1: CHECK RDKit AVAILABILITY
# ====================================================================================

try:
    import rdkit
    from rdkit import Chem

    print("=" * 100)
    print("RDKIT IS AVAILABLE")
    print("=" * 100)

    print("RDKit version:", rdkit.__version__)

except ImportError:
    print("=" * 100)
    print("RDKIT IS NOT INSTALLED")
    print("=" * 100)
    print("Install it using:")
    print("conda install -c conda-forge rdkit -y")

RDKIT IS AVAILABLE
RDKit version: 2023.09.6


In [21]:
# ====================================================================================
# CHEMICAL STRUCTURE NORMALIZATION AND CROSS-DATASET MAPPING
# DrugBank ↔ TWOSIDES
# ====================================================================================

import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi

print("=" * 100)
print("CHEMICAL STRUCTURE NORMALIZATION AND CROSS-DATASET MAPPING")
print("=" * 100)


# ====================================================================================
# 1. PREPARE UNIQUE DRUG STRUCTURE TABLES
# ====================================================================================

print("\n1. PREPARING UNIQUE DRUG STRUCTURES")
print("-" * 100)

# DrugBank
drugbank_structures = pd.concat(
    [
        drugbank_df[["ID1", "X1"]].rename(
            columns={"ID1": "drug_id", "X1": "raw_smiles"}
        ),
        drugbank_df[["ID2", "X2"]].rename(
            columns={"ID2": "drug_id", "X2": "raw_smiles"}
        )
    ],
    ignore_index=True
).drop_duplicates()

drugbank_structures = drugbank_structures.dropna(
    subset=["drug_id", "raw_smiles"]
)

drugbank_structures = drugbank_structures.drop_duplicates(
    subset=["drug_id"]
)

print("DrugBank unique drugs:", len(drugbank_structures))


# TWOSIDES
twosides_structures = pd.concat(
    [
        twosides_df[["ID1", "X1"]].rename(
            columns={"ID1": "drug_id", "X1": "raw_smiles"}
        ),
        twosides_df[["ID2", "X2"]].rename(
            columns={"ID2": "drug_id", "X2": "raw_smiles"}
        )
    ],
    ignore_index=True
).drop_duplicates()

twosides_structures = twosides_structures.dropna(
    subset=["drug_id", "raw_smiles"]
)

twosides_structures = twosides_structures.drop_duplicates(
    subset=["drug_id"]
)

print("TWOSIDES unique drugs:", len(twosides_structures))


# ====================================================================================
# 2. FUNCTION TO GENERATE NORMALIZED CHEMICAL IDENTIFIERS
# ====================================================================================

print("\n2. DEFINING CHEMICAL NORMALIZATION FUNCTION")
print("-" * 100)


def generate_structure_identifiers(smiles):
    """
    Generate multiple chemical representations for a SMILES string.

    Returns:
        canonical_smiles
        isomeric_smiles
        inchikey
        valid_structure
    """

    result = {
        "canonical_smiles": None,
        "isomeric_smiles": None,
        "inchikey": None,
        "valid_structure": False
    }

    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return result

        result["canonical_smiles"] = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

        result["isomeric_smiles"] = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

        result["inchikey"] = inchi.MolToInchiKey(mol)

        result["valid_structure"] = True

    except Exception:
        pass

    return result


# ====================================================================================
# 3. NORMALIZE DRUGBANK STRUCTURES
# ====================================================================================

print("\n3. NORMALIZING DRUGBANK STRUCTURES")
print("-" * 100)

drugbank_identifiers = drugbank_structures["raw_smiles"].apply(
    generate_structure_identifiers
)

drugbank_identifiers_df = pd.DataFrame(
    drugbank_identifiers.tolist()
)

drugbank_normalized = pd.concat(
    [
        drugbank_structures.reset_index(drop=True),
        drugbank_identifiers_df
    ],
    axis=1
)

print(
    "Valid DrugBank structures:",
    drugbank_normalized["valid_structure"].sum()
)

print(
    "Invalid DrugBank structures:",
    (~drugbank_normalized["valid_structure"]).sum()
)


# ====================================================================================
# 4. NORMALIZE TWOSIDES STRUCTURES
# ====================================================================================

print("\n4. NORMALIZING TWOSIDES STRUCTURES")
print("-" * 100)

twosides_identifiers = twosides_structures["raw_smiles"].apply(
    generate_structure_identifiers
)

twosides_identifiers_df = pd.DataFrame(
    twosides_identifiers.tolist()
)

twosides_normalized = pd.concat(
    [
        twosides_structures.reset_index(drop=True),
        twosides_identifiers_df
    ],
    axis=1
)

print(
    "Valid TWOSIDES structures:",
    twosides_normalized["valid_structure"].sum()
)

print(
    "Invalid TWOSIDES structures:",
    (~twosides_normalized["valid_structure"]).sum()
)


# ====================================================================================
# 5. RAW SMILES OVERLAP
# ====================================================================================

print("\n" + "=" * 100)
print("5. RAW SMILES OVERLAP")
print("-" * 100)

drugbank_raw_set = set(
    drugbank_normalized["raw_smiles"]
    .dropna()
    .astype(str)
)

twosides_raw_set = set(
    twosides_normalized["raw_smiles"]
    .dropna()
    .astype(str)
)

raw_overlap = drugbank_raw_set.intersection(
    twosides_raw_set
)

print("DrugBank unique raw SMILES:", len(drugbank_raw_set))
print("TWOSIDES unique raw SMILES:", len(twosides_raw_set))
print("Exact raw SMILES overlap:", len(raw_overlap))


# ====================================================================================
# 6. CANONICAL SMILES OVERLAP
# ====================================================================================

print("\n" + "=" * 100)
print("6. CANONICAL SMILES OVERLAP")
print("-" * 100)

drugbank_canonical_set = set(
    drugbank_normalized["canonical_smiles"]
    .dropna()
)

twosides_canonical_set = set(
    twosides_normalized["canonical_smiles"]
    .dropna()
)

canonical_overlap = drugbank_canonical_set.intersection(
    twosides_canonical_set
)

print(
    "DrugBank unique canonical SMILES:",
    len(drugbank_canonical_set)
)

print(
    "TWOSIDES unique canonical SMILES:",
    len(twosides_canonical_set)
)

print(
    "Canonical SMILES overlap:",
    len(canonical_overlap)
)


# ====================================================================================
# 7. ISOMERIC CANONICAL SMILES OVERLAP
# ====================================================================================

print("\n" + "=" * 100)
print("7. ISOMERIC CANONICAL SMILES OVERLAP")
print("-" * 100)

drugbank_isomeric_set = set(
    drugbank_normalized["isomeric_smiles"]
    .dropna()
)

twosides_isomeric_set = set(
    twosides_normalized["isomeric_smiles"]
    .dropna()
)

isomeric_overlap = drugbank_isomeric_set.intersection(
    twosides_isomeric_set
)

print(
    "DrugBank unique isomeric SMILES:",
    len(drugbank_isomeric_set)
)

print(
    "TWOSIDES unique isomeric SMILES:",
    len(twosides_isomeric_set)
)

print(
    "Isomeric SMILES overlap:",
    len(isomeric_overlap)
)


# ====================================================================================
# 8. INCHIKEY OVERLAP
# ====================================================================================

print("\n" + "=" * 100)
print("8. INCHIKEY OVERLAP")
print("-" * 100)

drugbank_inchikey_set = set(
    drugbank_normalized["inchikey"]
    .dropna()
)

twosides_inchikey_set = set(
    twosides_normalized["inchikey"]
    .dropna()
)

inchikey_overlap = drugbank_inchikey_set.intersection(
    twosides_inchikey_set
)

print(
    "DrugBank unique InChIKeys:",
    len(drugbank_inchikey_set)
)

print(
    "TWOSIDES unique InChIKeys:",
    len(twosides_inchikey_set)
)

print(
    "Exact InChIKey overlap:",
    len(inchikey_overlap)
)


# ====================================================================================
# 9. BUILD CANONICAL SMILES CROSSWALK
# ====================================================================================

print("\n" + "=" * 100)
print("9. CANONICAL SMILES CROSSWALK")
print("-" * 100)

canonical_crosswalk = (
    drugbank_normalized[
        ["drug_id", "canonical_smiles"]
    ]
    .dropna()
    .merge(
        twosides_normalized[
            ["drug_id", "canonical_smiles"]
        ].dropna(),
        on="canonical_smiles",
        how="inner",
        suffixes=("_drugbank", "_twosides")
    )
)

print(
    "Canonical SMILES mapping records:",
    len(canonical_crosswalk)
)

print(
    "DrugBank drugs mapped:",
    canonical_crosswalk[
        "drug_id_drugbank"
    ].nunique()
)

print(
    "TWOSIDES drugs mapped:",
    canonical_crosswalk[
        "drug_id_twosides"
    ].nunique()
)


# ====================================================================================
# 10. BUILD INCHIKEY CROSSWALK
# ====================================================================================

print("\n" + "=" * 100)
print("10. INCHIKEY CROSSWALK")
print("-" * 100)

inchikey_crosswalk = (
    drugbank_normalized[
        ["drug_id", "inchikey"]
    ]
    .dropna()
    .merge(
        twosides_normalized[
            ["drug_id", "inchikey"]
        ].dropna(),
        on="inchikey",
        how="inner",
        suffixes=("_drugbank", "_twosides")
    )
)

print(
    "InChIKey mapping records:",
    len(inchikey_crosswalk)
)

print(
    "DrugBank drugs mapped:",
    inchikey_crosswalk[
        "drug_id_drugbank"
    ].nunique()
)

print(
    "TWOSIDES drugs mapped:",
    inchikey_crosswalk[
        "drug_id_twosides"
    ].nunique()
)


# ====================================================================================
# 11. INCHIKEY MAPPING QUALITY
# ====================================================================================

print("\n" + "=" * 100)
print("11. INCHIKEY CROSSWALK QUALITY")
print("-" * 100)

if len(inchikey_crosswalk) > 0:

    db_to_tw = (
        inchikey_crosswalk
        .groupby("drug_id_drugbank")
        ["drug_id_twosides"]
        .nunique()
    )

    tw_to_db = (
        inchikey_crosswalk
        .groupby("drug_id_twosides")
        ["drug_id_drugbank"]
        .nunique()
    )

    print(
        "DrugBank → exactly one TWOSIDES drug:",
        (db_to_tw == 1).sum()
    )

    print(
        "DrugBank → multiple TWOSIDES drugs:",
        (db_to_tw > 1).sum()
    )

    print(
        "TWOSIDES → exactly one DrugBank drug:",
        (tw_to_db == 1).sum()
    )

    print(
        "TWOSIDES → multiple DrugBank drugs:",
        (tw_to_db > 1).sum()
    )

else:
    print("No InChIKey mappings found.")


# ====================================================================================
# 12. SAMPLE MAPPINGS
# ====================================================================================

print("\n" + "=" * 100)
print("12. SAMPLE INCHIKEY-BASED MAPPINGS")
print("-" * 100)

if len(inchikey_crosswalk) > 0:
    display(inchikey_crosswalk.head(20))


# ====================================================================================
# 13. FINAL COMPARISON
# ====================================================================================

print("\n" + "=" * 100)
print("FINAL STRUCTURE NORMALIZATION SUMMARY")
print("=" * 100)

summary_df = pd.DataFrame(
    {
        "Matching method": [
            "Raw SMILES",
            "Canonical SMILES",
            "Isomeric canonical SMILES",
            "Exact InChIKey"
        ],
        "Shared structures": [
            len(raw_overlap),
            len(canonical_overlap),
            len(isomeric_overlap),
            len(inchikey_overlap)
        ]
    }
)

display(summary_df)

print("\nDrugBank total drugs:", len(drugbank_normalized))
print("TWOSIDES total drugs:", len(twosides_normalized))

print("\nBest exact identifier-based mapping:")
print(
    "DrugBank drugs mapped through InChIKey:",
    inchikey_crosswalk["drug_id_drugbank"].nunique()
    if len(inchikey_crosswalk) > 0 else 0
)

print(
    "TWOSIDES drugs mapped through InChIKey:",
    inchikey_crosswalk["drug_id_twosides"].nunique()
    if len(inchikey_crosswalk) > 0 else 0
)

print("=" * 100)

CHEMICAL STRUCTURE NORMALIZATION AND CROSS-DATASET MAPPING

1. PREPARING UNIQUE DRUG STRUCTURES
----------------------------------------------------------------------------------------------------
DrugBank unique drugs: 1706
TWOSIDES unique drugs: 645

2. DEFINING CHEMICAL NORMALIZATION FUNCTION
----------------------------------------------------------------------------------------------------

3. NORMALIZING DRUGBANK STRUCTURES
----------------------------------------------------------------------------------------------------


[15:43:07] SMILES Parse Error: syntax error while parsing: OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1
[15:43:07] SMILES Parse Error: Failed parsing SMILES 'OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1' for input: 'OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1'
[15:43:07] bond type above 3 (17) is treated as unspecified!
[15:43:07] bond type above 3 (17) is treated as unspecified!
[15:43:07] Invalid InChI prefix in generating InChI Key
[15:43:07] bond type above 3 (17) is treated as unspecified!
[15:43:07] bond type above 3 (17) is treated as unspecified!
[15:43:07] Invalid InChI prefix in generating InChI Key
[15:43:07] bond type above 3 (17) is treated as unspecified!
[15:43:07] bond type above 3 (17) is treated as unspecified!
[15:43:07] Invali

Valid DrugBank structures: 1705
Invalid DrugBank structures: 1

4. NORMALIZING TWOSIDES STRUCTURES
----------------------------------------------------------------------------------------------------
Valid TWOSIDES structures: 645
Invalid TWOSIDES structures: 0

5. RAW SMILES OVERLAP
----------------------------------------------------------------------------------------------------
DrugBank unique raw SMILES: 1706
TWOSIDES unique raw SMILES: 645
Exact raw SMILES overlap: 7

6. CANONICAL SMILES OVERLAP
----------------------------------------------------------------------------------------------------
DrugBank unique canonical SMILES: 1678
TWOSIDES unique canonical SMILES: 645
Canonical SMILES overlap: 528

7. ISOMERIC CANONICAL SMILES OVERLAP
----------------------------------------------------------------------------------------------------
DrugBank unique isomeric SMILES: 1705
TWOSIDES unique isomeric SMILES: 645
Isomeric SMILES overlap: 287

8. INCHIKEY OVERLAP
--------------------

,drug_id_drugbank,inchikey,drug_id_twosides
0,DB00986,ANGKOCUUWGHLCE-UHFFFAOYSA-N,CID000003494
1,DB00376,HWHLPVGTWGOCJO-UHFFFAOYSA-N,CID000005572
2,DB01173,QVYRGXJJSLMXQH-UHFFFAOYSA-N,CID000004601
3,DB00809,BGDKAVGWHJFAGW-UHFFFAOYSA-N,CID000005593
4,DB00804,CURUTKGFNZGFSE-UHFFFAOYSA-N,CID000003042
5,DB00732,XXZSQOVSEBAPGS-UHFFFAOYSA-L,CID000047320
6,DB01062,XIQVNETUBQGFHX-UHFFFAOYSA-N,CID000004634
7,DB00967,JAUOIFJMECXRGI-UHFFFAOYSA-N,CID000124087
8,DB01390,UIIMBOGNXHQVGW-UHFFFAOYSA-M,CID000008953
9,DB00252,CXOFVDLJLONNDW-UHFFFAOYSA-N,CID000001775



FINAL STRUCTURE NORMALIZATION SUMMARY


,Matching method,Shared structures
0,Raw SMILES,7
1,Canonical SMILES,528
2,Isomeric canonical SMILES,287
3,Exact InChIKey,293



DrugBank total drugs: 1706
TWOSIDES total drugs: 645

Best exact identifier-based mapping:
DrugBank drugs mapped through InChIKey: 293
TWOSIDES drugs mapped through InChIKey: 293


In [22]:
# ====================================================================================
# MAPPING AUDIT AND CONFIDENCE CLASSIFICATION
# ====================================================================================

import pandas as pd

print("=" * 100)
print("DRUGBANK ↔ TWOSIDES MAPPING AUDIT")
print("=" * 100)


# ====================================================================================
# 1. CREATE PAIR IDENTIFIERS
# ====================================================================================

print("\n1. PREPARING MAPPING PAIRS")
print("-" * 100)

canonical_pairs = canonical_crosswalk[
    ["drug_id_drugbank", "drug_id_twosides"]
].drop_duplicates()

inchikey_pairs = inchikey_crosswalk[
    ["drug_id_drugbank", "drug_id_twosides"]
].drop_duplicates()

canonical_pair_set = set(
    zip(
        canonical_pairs["drug_id_drugbank"],
        canonical_pairs["drug_id_twosides"]
    )
)

inchikey_pair_set = set(
    zip(
        inchikey_pairs["drug_id_drugbank"],
        inchikey_pairs["drug_id_twosides"]
    )
)

print("Unique canonical mapping pairs:", len(canonical_pair_set))
print("Unique InChIKey mapping pairs:", len(inchikey_pair_set))


# ====================================================================================
# 2. COMPARE THE TWO MAPPING METHODS
# ====================================================================================

print("\n" + "=" * 100)
print("2. MAPPING METHOD COMPARISON")
print("-" * 100)

confirmed_by_both = canonical_pair_set.intersection(
    inchikey_pair_set
)

canonical_only = canonical_pair_set.difference(
    inchikey_pair_set
)

inchikey_only = inchikey_pair_set.difference(
    canonical_pair_set
)

print("Mappings confirmed by BOTH methods:", len(confirmed_by_both))
print("Canonical SMILES ONLY mappings:", len(canonical_only))
print("InChIKey ONLY mappings:", len(inchikey_only))


# ====================================================================================
# 3. CREATE A UNIFIED CROSSWALK
# ====================================================================================

print("\n" + "=" * 100)
print("3. BUILDING UNIFIED CROSSWALK")
print("-" * 100)

all_pairs = canonical_pair_set.union(inchikey_pair_set)

crosswalk_records = []

for db_id, tw_id in all_pairs:

    in_canonical = (db_id, tw_id) in canonical_pair_set
    in_inchikey = (db_id, tw_id) in inchikey_pair_set

    if in_canonical and in_inchikey:
        confidence = "HIGH_EXACT"
    elif in_inchikey:
        confidence = "HIGH_INCHIKEY"
    elif in_canonical:
        confidence = "MEDIUM_CANONICAL"
    else:
        confidence = "UNKNOWN"

    crosswalk_records.append(
        {
            "drugbank_id": db_id,
            "twosides_id": tw_id,
            "canonical_smiles_match": in_canonical,
            "inchikey_match": in_inchikey,
            "mapping_confidence": confidence
        }
    )

unified_crosswalk = pd.DataFrame(
    crosswalk_records
)

print("Total unique mapping pairs:", len(unified_crosswalk))

print("\nConfidence distribution:")
print(
    unified_crosswalk[
        "mapping_confidence"
    ].value_counts()
)


# ====================================================================================
# 4. CHECK DRUGBANK → TWOSIDES AMBIGUITY
# ====================================================================================

print("\n" + "=" * 100)
print("4. DRUGBANK → TWOSIDES MAPPING AMBIGUITY")
print("-" * 100)

db_mapping_counts = (
    unified_crosswalk
    .groupby("drugbank_id")["twosides_id"]
    .nunique()
)

print("DrugBank drugs with at least one mapping:", len(db_mapping_counts))

print(
    "DrugBank drugs mapping to exactly ONE TWOSIDES drug:",
    (db_mapping_counts == 1).sum()
)

print(
    "DrugBank drugs mapping to MULTIPLE TWOSIDES drugs:",
    (db_mapping_counts > 1).sum()
)

if (db_mapping_counts > 1).sum() > 0:

    ambiguous_db_ids = db_mapping_counts[
        db_mapping_counts > 1
    ].index.tolist()

    print("\nExample ambiguous DrugBank mappings:")

    display(
        unified_crosswalk[
            unified_crosswalk["drugbank_id"]
            .isin(ambiguous_db_ids[:10])
        ]
        .sort_values("drugbank_id")
    )


# ====================================================================================
# 5. CHECK TWOSIDES → DRUGBANK AMBIGUITY
# ====================================================================================

print("\n" + "=" * 100)
print("5. TWOSIDES → DRUGBANK MAPPING AMBIGUITY")
print("-" * 100)

tw_mapping_counts = (
    unified_crosswalk
    .groupby("twosides_id")["drugbank_id"]
    .nunique()
)

print("TWOSIDES drugs with at least one mapping:", len(tw_mapping_counts))

print(
    "TWOSIDES drugs mapping to exactly ONE DrugBank drug:",
    (tw_mapping_counts == 1).sum()
)

print(
    "TWOSIDES drugs mapping to MULTIPLE DrugBank drugs:",
    (tw_mapping_counts > 1).sum()
)

if (tw_mapping_counts > 1).sum() > 0:

    ambiguous_tw_ids = tw_mapping_counts[
        tw_mapping_counts > 1
    ].index.tolist()

    print("\nExample ambiguous TWOSIDES mappings:")

    display(
        unified_crosswalk[
            unified_crosswalk["twosides_id"]
            .isin(ambiguous_tw_ids[:10])
        ]
        .sort_values("twosides_id")
    )


# ====================================================================================
# 6. TWOSIDES COVERAGE
# ====================================================================================

print("\n" + "=" * 100)
print("6. TWOSIDES COVERAGE")
print("-" * 100)

all_twosides_ids = set(
    twosides_normalized["drug_id"]
)

mapped_twosides_ids = set(
    unified_crosswalk["twosides_id"]
)

unmapped_twosides_ids = (
    all_twosides_ids -
    mapped_twosides_ids
)

print("Total TWOSIDES drugs:", len(all_twosides_ids))
print("Mapped TWOSIDES drugs:", len(mapped_twosides_ids))
print("Unmapped TWOSIDES drugs:", len(unmapped_twosides_ids))

print(
    "TWOSIDES mapping coverage:",
    round(
        len(mapped_twosides_ids)
        / len(all_twosides_ids)
        * 100,
        2
    ),
    "%"
)


# ====================================================================================
# 7. DRUGBANK COVERAGE
# ====================================================================================

print("\n" + "=" * 100)
print("7. DRUGBANK COVERAGE")
print("-" * 100)

all_drugbank_ids = set(
    drugbank_normalized["drug_id"]
)

mapped_drugbank_ids = set(
    unified_crosswalk["drugbank_id"]
)

unmapped_drugbank_ids = (
    all_drugbank_ids -
    mapped_drugbank_ids
)

print("Total DrugBank drugs:", len(all_drugbank_ids))
print("Mapped DrugBank drugs:", len(mapped_drugbank_ids))
print("Unmapped DrugBank drugs:", len(unmapped_drugbank_ids))

print(
    "DrugBank mapping coverage:",
    round(
        len(mapped_drugbank_ids)
        / len(all_drugbank_ids)
        * 100,
        2
    ),
    "%"
)


# ====================================================================================
# 8. EXAMINE CANONICAL-ONLY MAPPINGS
# ====================================================================================

print("\n" + "=" * 100)
print("8. CANONICAL-ONLY MAPPING INSPECTION")
print("-" * 100)

canonical_only_df = pd.DataFrame(
    list(canonical_only),
    columns=["drugbank_id", "twosides_id"]
)

print(
    "Canonical-only mapping pairs:",
    len(canonical_only_df)
)

if len(canonical_only_df) > 0:

    canonical_only_details = (
        canonical_only_df
        .merge(
            drugbank_normalized[
                [
                    "drug_id",
                    "raw_smiles",
                    "canonical_smiles",
                    "isomeric_smiles",
                    "inchikey"
                ]
            ],
            left_on="drugbank_id",
            right_on="drug_id",
            how="left"
        )
        .drop(columns=["drug_id"])
        .rename(
            columns={
                "raw_smiles": "drugbank_raw_smiles",
                "canonical_smiles": "drugbank_canonical",
                "isomeric_smiles": "drugbank_isomeric",
                "inchikey": "drugbank_inchikey"
            }
        )
        .merge(
            twosides_normalized[
                [
                    "drug_id",
                    "raw_smiles",
                    "canonical_smiles",
                    "isomeric_smiles",
                    "inchikey"
                ]
            ],
            left_on="twosides_id",
            right_on="drug_id",
            how="left"
        )
        .drop(columns=["drug_id"])
        .rename(
            columns={
                "raw_smiles": "twosides_raw_smiles",
                "canonical_smiles": "twosides_canonical",
                "isomeric_smiles": "twosides_isomeric",
                "inchikey": "twosides_inchikey"
            }
        )
    )

    print("\nFirst 20 canonical-only cases:")
    display(canonical_only_details.head(20))


# ====================================================================================
# 9. CREATE HIGH-CONFIDENCE CROSSWALK
# ====================================================================================

print("\n" + "=" * 100)
print("9. HIGH-CONFIDENCE CROSSWALK")
print("-" * 100)

high_confidence_crosswalk = unified_crosswalk[
    unified_crosswalk["mapping_confidence"]
    .isin([
        "HIGH_EXACT",
        "HIGH_INCHIKEY"
    ])
].copy()

print(
    "High-confidence mapping pairs:",
    len(high_confidence_crosswalk)
)

print(
    "Unique DrugBank drugs:",
    high_confidence_crosswalk[
        "drugbank_id"
    ].nunique()
)

print(
    "Unique TWOSIDES drugs:",
    high_confidence_crosswalk[
        "twosides_id"
    ].nunique()
)


# ====================================================================================
# 10. SAVE CROSSWALK FILES
# ====================================================================================

print("\n" + "=" * 100)
print("10. SAVING CROSSWALK RESULTS")
print("-" * 100)

crosswalk_output_dir = (
    project_root /
    "data" /
    "interim" /
    "normalized"
)

crosswalk_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

unified_crosswalk_path = (
    crosswalk_output_dir /
    "drugbank_twosides_crosswalk_all.csv"
)

high_confidence_path = (
    crosswalk_output_dir /
    "drugbank_twosides_crosswalk_high_confidence.csv"
)

canonical_only_path = (
    crosswalk_output_dir /
    "drugbank_twosides_crosswalk_canonical_only.csv"
)

unified_crosswalk.to_csv(
    unified_crosswalk_path,
    index=False
)

high_confidence_crosswalk.to_csv(
    high_confidence_path,
    index=False
)

canonical_only_df.to_csv(
    canonical_only_path,
    index=False
)

print("Saved:")
print("  All mappings:", unified_crosswalk_path)
print("  High-confidence mappings:", high_confidence_path)
print("  Canonical-only mappings:", canonical_only_path)


# ====================================================================================
# 11. FINAL AUDIT SUMMARY
# ====================================================================================

print("\n" + "=" * 100)
print("FINAL MAPPING AUDIT SUMMARY")
print("=" * 100)

print(
    "Total unique mapping pairs:",
    len(unified_crosswalk)
)

print(
    "High-confidence mappings:",
    len(high_confidence_crosswalk)
)

print(
    "Canonical-only mappings:",
    len(canonical_only_df)
)

print(
    "Mapped DrugBank drugs:",
    len(mapped_drugbank_ids),
    "/",
    len(all_drugbank_ids)
)

print(
    "Mapped TWOSIDES drugs:",
    len(mapped_twosides_ids),
    "/",
    len(all_twosides_ids)
)

print(
    "DrugBank → multiple TWOSIDES:",
    (db_mapping_counts > 1).sum()
)

print(
    "TWOSIDES → multiple DrugBank:",
    (tw_mapping_counts > 1).sum()
)

print("\nCrosswalk files have been created successfully.")

print("=" * 100)

DRUGBANK ↔ TWOSIDES MAPPING AUDIT

1. PREPARING MAPPING PAIRS
----------------------------------------------------------------------------------------------------
Unique canonical mapping pairs: 548
Unique InChIKey mapping pairs: 293

2. MAPPING METHOD COMPARISON
----------------------------------------------------------------------------------------------------
Mappings confirmed by BOTH methods: 287
Canonical SMILES ONLY mappings: 261
InChIKey ONLY mappings: 6

3. BUILDING UNIFIED CROSSWALK
----------------------------------------------------------------------------------------------------
Total unique mapping pairs: 554

Confidence distribution:
mapping_confidence
HIGH_EXACT          287
MEDIUM_CANONICAL    261
HIGH_INCHIKEY         6
Name: count, dtype: int64

4. DRUGBANK → TWOSIDES MAPPING AMBIGUITY
----------------------------------------------------------------------------------------------------
DrugBank drugs with at least one mapping: 554
DrugBank drugs mapping to exactly ONE

,drugbank_id,twosides_id,canonical_smiles_match,inchikey_match,mapping_confidence
336,DB00468,CID000001065,True,False,MEDIUM_CANONICAL
302,DB00908,CID000001065,True,False,MEDIUM_CANONICAL
82,DB13139,CID000002083,True,False,MEDIUM_CANONICAL
123,DB01001,CID000002083,True,True,HIGH_EXACT
391,DB01002,CID000002474,True,False,MEDIUM_CANONICAL
540,DB00297,CID000002474,True,True,HIGH_EXACT
315,DB00341,CID000002678,True,True,HIGH_EXACT
493,DB06282,CID000002678,True,False,MEDIUM_CANONICAL
226,DB01175,CID000002771,True,False,MEDIUM_CANONICAL
274,DB00215,CID000002771,True,True,HIGH_EXACT



6. TWOSIDES COVERAGE
----------------------------------------------------------------------------------------------------
Total TWOSIDES drugs: 645
Mapped TWOSIDES drugs: 534
Unmapped TWOSIDES drugs: 111
TWOSIDES mapping coverage: 82.79 %

7. DRUGBANK COVERAGE
----------------------------------------------------------------------------------------------------
Total DrugBank drugs: 1706
Mapped DrugBank drugs: 554
Unmapped DrugBank drugs: 1152
DrugBank mapping coverage: 32.47 %

8. CANONICAL-ONLY MAPPING INSPECTION
----------------------------------------------------------------------------------------------------
Canonical-only mapping pairs: 261

First 20 canonical-only cases:


,drugbank_id,twosides_id,drugbank_raw_smiles,drugbank_canonical,drugbank_isomeric,drugbank_inchikey,twosides_raw_smiles,twosides_canonical,twosides_isomeric,twosides_inchikey
0,DB00764,CID000123620,[H][C@@]12C[C@@H](C)[C@](O)(C(=O)CCl)[C@@]1(C)...,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(Cl)C(O)CC2(C)C1(...,C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C...,QLIIKPVHVRXHRI-CXSFZGCWSA-N,CC1CC2C3CCC4=CC(=O)C=CC4(C3(C(CC2(C1(C(=O)CCl)...,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(Cl)C(O)CC2(C)C1(...,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(Cl)C(O)CC2(C)C1(...,QLIIKPVHVRXHRI-UHFFFAOYSA-N
1,DB00222,CID000003476,CCC1=C(C)CN(C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=...,CCC1=C(C)CN(C(=O)NCCc2ccc(S(=O)(=O)NC(=O)NC3CC...,CCC1=C(C)CN(C(=O)NCCc2ccc(S(=O)(=O)NC(=O)N[C@H...,WIGIZIANZCJQQY-RUCARUNLSA-N,CCC1=C(CN(C1=O)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)N...,CCC1=C(C)CN(C(=O)NCCc2ccc(S(=O)(=O)NC(=O)NC3CC...,CCC1=C(C)CN(C(=O)NCCc2ccc(S(=O)(=O)NC(=O)NC3CC...,WIGIZIANZCJQQY-UHFFFAOYSA-N
2,DB01623,CID000005454,CN(C)S(=O)(=O)C1=CC2=C(SC3=CC=CC=C3\C2=C\CCN2C...,CN1CCN(CCC=C2c3ccccc3Sc3ccc(S(=O)(=O)N(C)C)cc3...,CN1CCN(CC/C=C2/c3ccccc3Sc3ccc(S(=O)(=O)N(C)C)c...,GFBKORZTTCHDGY-UWVJOHFNSA-N,CN1CCN(CC1)CCC=C2C3=CC=CC=C3SC4=C2C=C(C=C4)S(=...,CN1CCN(CCC=C2c3ccccc3Sc3ccc(S(=O)(=O)N(C)C)cc3...,CN1CCN(CCC=C2c3ccccc3Sc3ccc(S(=O)(=O)N(C)C)cc3...,GFBKORZTTCHDGY-UHFFFAOYSA-N
3,DB00175,CID000004889,[H][C@]12[C@H](C[C@H](O)C=C1C=C[C@H](C)[C@@H]2...,CCC(C)C(=O)OC1CC(O)C=C2C=CC(C)C(CCC(O)CC(O)CC(...,CC[C@H](C)C(=O)O[C@H]1C[C@H](O)C=C2C=C[C@H](C)...,TUZYXOIXSAXUGO-PZAWKZKUSA-N,CCC(C)C(=O)OC1CC(C=C2C1C(C(C=C2)C)CCC(CC(CC(=O...,CCC(C)C(=O)OC1CC(O)C=C2C=CC(C)C(CCC(O)CC(O)CC(...,CCC(C)C(=O)OC1CC(O)C=C2C=CC(C)C(CCC(O)CC(O)CC(...,TUZYXOIXSAXUGO-UHFFFAOYSA-N
4,DB00747,CID000005184,CN1[C@H]2C[C@@H](C[C@@H]1[C@H]1O[C@@H]21)OC(=O...,CN1C2CC(OC(=O)C(CO)c3ccccc3)CC1C1OC12,CN1[C@@H]2C[C@@H](OC(=O)[C@H](CO)c3ccccc3)C[C@...,STECJAGHUSJQJN-FWXGHANASA-N,CN1C2CC(CC1C3C2O3)OC(=O)C(CO)C4=CC=CC=C4,CN1C2CC(OC(=O)C(CO)c3ccccc3)CC1C1OC12,CN1C2CC(OC(=O)C(CO)c3ccccc3)CC1C1OC12,STECJAGHUSJQJN-UHFFFAOYSA-N
5,DB00584,CID000003222,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@@H](C)C(=O)N1C...,CCOC(=O)C(CCc1ccccc1)NC(C)C(=O)N1CCCC1C(=O)O,CCOC(=O)[C@H](CCc1ccccc1)N[C@@H](C)C(=O)N1CCC[...,GBXSMTUPTTWBMN-XIRDDKMYSA-N,CCOC(=O)C(CCC1=CC=CC=C1)NC(C)C(=O)N2CCCC2C(=O)O,CCOC(=O)C(CCc1ccccc1)NC(C)C(=O)N1CCCC1C(=O)O,CCOC(=O)C(CCc1ccccc1)NC(C)C(=O)N1CCCC1C(=O)O,GBXSMTUPTTWBMN-UHFFFAOYSA-N
6,DB00227,CID000003962,[H][C@]12[C@H](C[C@@H](C)C=C1C=C[C@H](C)[C@@H]...,CCC(C)C(=O)OC1CC(C)C=C2C=CC(C)C(CCC3CC(O)CC(=O...,CC[C@H](C)C(=O)O[C@H]1C[C@@H](C)C=C2C=C[C@H](C...,PCZOHLXUXFIOCF-BXMDZJJMSA-N,CCC(C)C(=O)OC1CC(C=C2C1C(C(C=C2)C)CCC3CC(CC(=O...,CCC(C)C(=O)OC1CC(C)C=C2C=CC(C)C(CCC3CC(O)CC(=O...,CCC(C)C(=O)OC1CC(C)C=C2C=CC(C)C(CCC3CC(O)CC(=O...,PCZOHLXUXFIOCF-UHFFFAOYSA-N
7,DB00570,CID000013342,[H][C@@]12N(C)C3=CC(OC)=C(C=C3[C@@]11CCN3CC=C[...,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,CC[C@]1(O)C[C@@H]2CN(CCc3c([nH]c4ccccc34)[C@@]...,JXLYSJRDGCGARV-CFWMRBGOSA-N,CCC1(CC2CC(C3=C(CCN(C2)C1)C4=CC=CC=C4N3)(C5=C(...,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,JXLYSJRDGCGARV-UHFFFAOYSA-N
8,DB01030,CID000005515,CC[C@@]1(O)C(=O)OCC2=C1C=C1N(CC3=CC4=C(C=CC(O)...,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,CC[C@@]1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C...,UCFGDBYHRUNTLO-QHCPKHFHSA-N,CCC1(C2=C(COC1=O)C(=O)N3CC4=CC5=C(C=CC(=C5CN(C...,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,UCFGDBYHRUNTLO-UHFFFAOYSA-N
9,DB00844,CID000004419,O[C@H]1CC[C@@]2(O)[C@H]3CC4=CC=C(O)C5=C4[C@@]2...,Oc1ccc2c3c1OC1C(O)CCC4(O)C(C2)N(CC2CCC2)CCC314,Oc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@@]4(O)[C@@H](C...,NETZHAKZCGBWSS-CEDHKZHLSA-N,C1CC(C1)CN2CCC34C5C(CCC3(C2CC6=C4C(=C(C=C6)O)O...,Oc1ccc2c3c1OC1C(O)CCC4(O)C(C2)N(CC2CCC2)CCC314,Oc1ccc2c3c1OC1C(O)CCC4(O)C(C2)N(CC2CCC2)CCC314,NETZHAKZCGBWSS-UHFFFAOYSA-N



9. HIGH-CONFIDENCE CROSSWALK
----------------------------------------------------------------------------------------------------
High-confidence mapping pairs: 293
Unique DrugBank drugs: 293
Unique TWOSIDES drugs: 293

10. SAVING CROSSWALK RESULTS
----------------------------------------------------------------------------------------------------


NameError: name 'project_root' is not defined

In [23]:
# ====================================================================================
# FIX PROJECT ROOT AND SAVE CROSSWALK RESULTS
# ====================================================================================

from pathlib import Path
import json

print("=" * 100)
print("SAVING DRUGBANK ↔ TWOSIDES CROSSWALK RESULTS")
print("=" * 100)


# ====================================================================================
# 1. DEFINE PROJECT ROOT AGAIN
# ====================================================================================

current_dir = Path.cwd().resolve()

# We are currently inside:
# Prescription_safety_graph/notebooks
# So the parent is the project root

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    # Fallback: search upward for project root
    project_root = current_dir

    for parent in [current_dir] + list(current_dir.parents):
        if (parent / "data").exists():
            project_root = parent
            break


print("\nCurrent directory:")
print(current_dir)

print("\nDetected project root:")
print(project_root)


# ====================================================================================
# 2. CREATE OUTPUT DIRECTORY
# ====================================================================================

crosswalk_output_dir = (
    project_root /
    "data" /
    "interim" /
    "normalized"
)

crosswalk_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("\nOutput directory:")
print(crosswalk_output_dir)

print("Directory exists:", crosswalk_output_dir.exists())


# ====================================================================================
# 3. CHECK REQUIRED VARIABLES
# ====================================================================================

required_variables = [
    "unified_crosswalk",
    "high_confidence_crosswalk"
]

print("\n" + "=" * 100)
print("VARIABLE AVAILABILITY CHECK")
print("=" * 100)

for variable_name in required_variables:

    if variable_name in globals():
        print(f"✓ {variable_name} is available")

        variable = globals()[variable_name]

        if hasattr(variable, "shape"):
            print(f"  Shape: {variable.shape}")

    else:
        print(f"✗ {variable_name} is NOT available")


# ====================================================================================
# 4. STOP IF THE REQUIRED DATAFRAMES ARE MISSING
# ====================================================================================

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:

    print("\n" + "=" * 100)
    print("CANNOT SAVE YET")
    print("=" * 100)

    print(
        "The following variables are missing from notebook memory:"
    )

    for variable_name in missing_variables:
        print("-", variable_name)

    print(
        "\nDo NOT rerun random cells. "
        "Rerun the mapping audit cell that creates these variables first."
    )

else:

    # ====================================================================================
    # 5. SAVE FULL UNIFIED CROSSWALK
    # ====================================================================================

    print("\n" + "=" * 100)
    print("SAVING FULL UNIFIED CROSSWALK")
    print("=" * 100)

    unified_crosswalk_path = (
        crosswalk_output_dir /
        "drugbank_twosides_unified_crosswalk.csv"
    )

    unified_crosswalk.to_csv(
        unified_crosswalk_path,
        index=False
    )

    print("Saved:")
    print(unified_crosswalk_path)

    print("Rows:", len(unified_crosswalk))
    print("Columns:", list(unified_crosswalk.columns))


    # ====================================================================================
    # 6. SAVE HIGH-CONFIDENCE CROSSWALK
    # ====================================================================================

    print("\n" + "=" * 100)
    print("SAVING HIGH-CONFIDENCE CROSSWALK")
    print("=" * 100)

    high_confidence_path = (
        crosswalk_output_dir /
        "drugbank_twosides_high_confidence_crosswalk.csv"
    )

    high_confidence_crosswalk.to_csv(
        high_confidence_path,
        index=False
    )

    print("Saved:")
    print(high_confidence_path)

    print("Rows:", len(high_confidence_crosswalk))
    print("Columns:", list(high_confidence_crosswalk.columns))


    # ====================================================================================
    # 7. SAVE MAPPING SUMMARY
    # ====================================================================================

    print("\n" + "=" * 100)
    print("SAVING MAPPING SUMMARY")
    print("=" * 100)

    summary_data = {

        "drugbank_total_drugs":
            int(len(drugbank_ids)),

        "twosides_total_drugs":
            int(len(twosides_ids)),

        "total_unified_mapping_pairs":
            int(len(unified_crosswalk)),

        "high_confidence_mapping_pairs":
            int(len(high_confidence_crosswalk)),

        "mapped_drugbank_drugs_unified":
            int(
                unified_crosswalk[
                    "drugbank_id"
                ].nunique()
            ),

        "mapped_twosides_drugs_unified":
            int(
                unified_crosswalk[
                    "twosides_id"
                ].nunique()
            ),

        "mapped_drugbank_drugs_high_confidence":
            int(
                high_confidence_crosswalk[
                    "drugbank_id"
                ].nunique()
            ),

        "mapped_twosides_drugs_high_confidence":
            int(
                high_confidence_crosswalk[
                    "twosides_id"
                ].nunique()
            ),

        "confidence_distribution":
            {
                str(key): int(value)
                for key, value in
                unified_crosswalk[
                    "mapping_confidence"
                ]
                .value_counts()
                .to_dict()
                .items()
            }
    }


    summary_path = (
        crosswalk_output_dir /
        "drugbank_twosides_mapping_summary.json"
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            summary_data,
            file,
            indent=4
        )


    print("Saved:")
    print(summary_path)


    # ====================================================================================
    # 8. VERIFY SAVED FILES
    # ====================================================================================

    print("\n" + "=" * 100)
    print("FINAL FILE VERIFICATION")
    print("=" * 100)

    output_files = [
        unified_crosswalk_path,
        high_confidence_path,
        summary_path
    ]

    for file_path in output_files:

        print(
            f"\n{file_path.name}"
        )

        print(
            "Exists:",
            file_path.exists()
        )

        if file_path.exists():

            file_size_mb = (
                file_path.stat().st_size /
                (1024 * 1024)
            )

            print(
                f"Size: {file_size_mb:.4f} MB"
            )


    # ====================================================================================
    # 9. FINAL SUMMARY
    # ====================================================================================

    print("\n" + "=" * 100)
    print("CROSSWALK SAVING COMPLETED SUCCESSFULLY")
    print("=" * 100)

    print(
        f"\nFull unified crosswalk: "
        f"{len(unified_crosswalk)} mapping pairs"
    )

    print(
        f"High-confidence crosswalk: "
        f"{len(high_confidence_crosswalk)} mapping pairs"
    )

    print(
        "\nSaved location:"
    )

    print(crosswalk_output_dir)

    print(
        "\nFiles created:"
    )

    for file_path in output_files:
        print("-", file_path.name)

    print("=" * 100)

SAVING DRUGBANK ↔ TWOSIDES CROSSWALK RESULTS

Current directory:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/notebooks

Detected project root:
/Users/apple/Documents/SIH2026/Prescription_safety_graph

Output directory:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/interim/normalized
Directory exists: True

VARIABLE AVAILABILITY CHECK
✓ unified_crosswalk is available
  Shape: (554, 5)
✓ high_confidence_crosswalk is available
  Shape: (293, 5)

SAVING FULL UNIFIED CROSSWALK
Saved:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/interim/normalized/drugbank_twosides_unified_crosswalk.csv
Rows: 554
Columns: ['drugbank_id', 'twosides_id', 'canonical_smiles_match', 'inchikey_match', 'mapping_confidence']

SAVING HIGH-CONFIDENCE CROSSWALK
Saved:
/Users/apple/Documents/SIH2026/Prescription_safety_graph/data/interim/normalized/drugbank_twosides_high_confidence_crosswalk.csv
Rows: 293
Columns: ['drugbank_id', 'twosides_id', 'canonical_smiles_match', 'inc